In [1]:
import cv2
import numpy as np
import argparse
import time
import os
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

In [ ]:
CONFIG = {
    "model_path": "yolov8n.pt",

    "confidence_threshold": 0.5,
    "iou_threshold": 0.45,

    "violation_save_dir": "violations",
 
    "show_fps": True,

    "auto_save_violations": True,

    "class_names_custom": {
        0: "with_helmet",
        1: "without_helmet",
        2: "rider",
    },

    "colors": {
        "with_helmet":    (0, 200, 0),      # Xanh lá
        "without_helmet": (0, 0, 255),      # Đỏ
        "rider":          (255, 165, 0),    # Cam
        "violation_box":  (0, 0, 255),      # Đỏ
        "safe_box":       (0, 200, 0),      # Xanh
        "person":         (255, 200, 0),    # Vàng
    },
}

In [3]:
from detec import PhatHienKhongMu

In [ ]:
def run_detection(source, model_path=None, show=True, save_output=False):
    """Chạy detection trên source (webcam / video / ảnh / folder)."""
 
    detector = detec(
        model_path=model_path,
        conf=CONFIG["confidence_threshold"],
        iou=CONFIG["iou_threshold"],
    )
 
    source_str = str(source)
    is_image   = source_str.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    is_webcam  = source_str.isdigit()
 
    if is_image:
        _process_image(detector, source_str, show)
        return
 
    cap_src = int(source) if is_webcam else source_str
    cap     = cv2.VideoCapture(cap_src)
    if not cap.isOpened():
        print(f"[ERROR] Không thể mở: {source}")
        return
 
    writer = None
    if save_output:
        fps_in = cap.get(cv2.CAP_PROP_FPS) or 30
        w_in   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h_in   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_path = f"output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp4"
        writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (w_in, h_in))
        print(f"[INFO] Lưu output: {out_path}")
 
    print("[INFO] Bắt đầu. Nhấn 'q' để thoát, 's' để lưu frame hiện tại.")
 
    prev_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            print("[INFO] Hết video / không đọc được frame.")
            break
 
        now = time.time()
        detector._fps = 1.0 / (now - prev_time + 1e-6)
        prev_time     = now
 
        annotated, violations = detector.process_frame(frame)
 
        if CONFIG["auto_save_violations"] and violations:
            detector.save_violation(annotated, violations, detector.total_frames)
 
        if writer:
            writer.write(annotated)
 
        if show:
            cv2.imshow("Helmet Violation Detector", annotated)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("s"):
                detector.save_violation(annotated, violations, detector.total_frames)
 
    cap.release()
    if writer:
        writer.release()
    cv2.destroyAllWindows()
 
    print(f"\n[DONE] Tổng frames: {detector.total_frames}")
    print(f"[DONE] Tổng vi phạm phát hiện: {detector.total_violations}")

In [ ]:
def _process_image(detector, path, show=True):
    frame = cv2.imread(path)
    if frame is None:
        print(f"[ERROR] Không đọc được ảnh: {path}")
        return
 
    annotated, violations = detector.process_frame(frame)
    print(f"[RESULT] Vi phạm: {len(violations)}")
 
    out_path = f"result_{Path(path).stem}.jpg"
    cv2.imwrite(out_path, annotated)
    print(f"[SAVE] Kết quả: {out_path}")
 
    if show:
        cv2.imshow("Helmet Violation Detector", annotated)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

In [6]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Phát hiện vi phạm không đội mũ bảo hiểm")
    parser.add_argument("--source",  default="0",         help="Nguồn: 0=webcam, path video/ảnh, rtsp://...")
    parser.add_argument("--model",   default=None,        help="Đường dẫn model (.pt). Mặc định: yolov8n.pt")
    parser.add_argument("--conf",    type=float, default=0.5, help="Ngưỡng tin cậy")
    parser.add_argument("--no-show", action="store_true", help="Không hiển thị cửa sổ")
    parser.add_argument("--save",    action="store_true", help="Lưu video output")
    args = parser.parse_args()
 
    CONFIG["confidence_threshold"] = args.conf
 
    run_detection(
        source=args.source,
        model_path=args.model,
        show=not args.no_show,
        save_output=args.save,
    )

usage: ipykernel_launcher.py [-h] [--source SOURCE] [--model MODEL]
                             [--conf CONF] [--no-show] [--save]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\ADMIN\AppData\Roaming\jupyter\runtime\kernel-v3670f7ce46b7b17365a851dea52d8c37ddda54241.json


SystemExit: 2

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3554: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
